# Notebook 3: Architecture & The Model

This notebook covers SGET's architecture: the model-signal-view pattern, the central `SceneGraphModel`, and the layer configuration system. After this notebook you'll understand how data flows through the application.

## Prerequisites
- Notebooks 1 and 2 completed
- Neo4j running with the example DSG loaded (Notebook 2 does this)

## 1. Architecture Overview

SGET follows a **model-signal-view** pattern (Qt's version of MVC):

```
 JSON file ──► spark_dsg ──► heracles bulk load ──► Neo4j
                                                      │
                                              SceneGraphModel
                                           (cache + Qt signals)
                                        /      |       |        \
                                  GraphView  LayerPanel  PropertyPanel  SnapshotPanel
                                  (2D view)  (toggles)   (editor)      (save/restore)
```

**Key principle**: all data flows through the `SceneGraphModel`. Views never talk to Neo4j directly — they read from the model's in-memory cache and react to Qt signals when data changes.

**Why a cache?** Neo4j queries are fast, but not fast enough for rendering hundreds of nodes at 60fps. The model maintains a dict of node properties and a list of edges, populated from Neo4j on load. Mutations update both the cache and Neo4j, then emit signals.

**Why not keep spark_dsg in memory?** spark_dsg's pybind11 bindings are designed for bulk construction and serialization, not fine-grained mutation with change notification. The DSG object is transient — used only at the load/save boundaries.

<video src="media/ArchitectureOverview.mp4" controls width="720">
  Your browser does not support the video tag.
</video>

## 2. The SceneGraphModel

The `SceneGraphModel` (a `QObject`) is the heart of the application. Let's see it in action — we'll create one, connect it to Neo4j, load a graph, and observe how signals fire.

<video src="media/ModelSignalView.mp4" controls width="720">
  Your browser does not support the video tag.
</video>

In [ ]:
# Qt requires a QApplication instance for signals to work, even without a GUI.
import os

from heracles.utils import get_labelspace
from PySide6.QtWidgets import QApplication

from sget.backend.scene_graph_model import SceneGraphModel

app = QApplication.instance() or QApplication([])

model = SceneGraphModel()
model.connect("neo4j://127.0.0.1:7687", "neo4j", "neo4j_pw")

# Set labelspaces (same as app.py does from CLI args).
object_labels = get_labelspace("ade20k_mit_label_space.yaml")
room_labels = get_labelspace("b45_label_space.yaml")
model.set_labelspaces(
    object_labels={v: k for k, v in object_labels.items()},
    room_labels={v: k for k, v in room_labels.items()},
)

print(f"Connected: {model.connected}")

In [ ]:
# SignalSpy — a simple helper that records every emission of a Qt signal.
# This is the same pattern used in our test suite (tests/test_scene_graph_model.py).
class SignalSpy:
    def __init__(self, signal):
        self.calls = []
        signal.connect(lambda *args: self.calls.append(args))


# Watch the graph_loaded signal.
loaded_spy = SignalSpy(model.graph_loaded)

# Load the example DSG.
DSG_PATH = os.path.expanduser(
    "~/software/mit/sget/heracles/heracles/examples/scene_graphs/example_dsg.json"
)
model.load_from_json(DSG_PATH)

print(f"graph_loaded fired: {len(loaded_spy.calls)} time(s)")
print(f"Cache: {len(model.get_all_nodes())} nodes, {len(model.get_edges())} edges")

In [ ]:
# Now watch what happens when we add a node — the model updates the cache,
# writes to Neo4j, and emits node_added.
from heracles import constants

add_spy = SignalSpy(model.node_added)

model.add_node(
    constants.OBJECTS,
    "o(99)",
    {
        "pos_x": 0.0,
        "pos_y": 0.0,
        "pos_z": 0.0,
        "bbox_x": 0.0,
        "bbox_y": 0.0,
        "bbox_z": 0.0,
        "bbox_l": 1.0,
        "bbox_w": 1.0,
        "bbox_h": 1.0,
        "class": "box",
        "name": "demo_node",
    },
)

print(f"node_added fired: {add_spy.calls}")
print(f"Node in cache: {model.get_node('o(99)') is not None}")
print(f"Node layer:    {model.get_node_layer('o(99)')}")

# Clean up.
model.remove_node("o(99)")

## 3. Layer Configuration — Single Source of Truth

SGET uses `utils/colors.py` as the **single source of truth** for layer configuration. It defines `LAYER_STYLES` — an ordered list of `LayerStyle` dataclasses that specify each layer's label, ID, display name, color, and category characters.

Everything else derives from this list: the model's `LAYER_ORDER`, the layer panel's rows, the graph view's node colors, and the spatial layout.

**Important**: the position in `LAYER_STYLES` defines the hierarchy order, NOT the raw layer IDs. MeshPlaces has `layer_id=20` but sits below Rooms (`layer_id=4`) in the hierarchy. Code that needs to compare hierarchy levels (e.g., detecting parent-child relationships) uses the index in `LAYER_STYLES`.

**Multi-char categories**: each layer has a `category_chars` tuple listing all known NodeSymbol prefixes. For example, MeshPlaces supports both `'P'` (Place2dNodeAttributes) and `'t'` (TravNodeAttributes). The first character in the tuple is the default for new node creation.

All widget files use `heracles.constants` (e.g., `hc.ROOMS`, `hc.OBJECTS`) for layer label strings instead of hardcoded `"Room"`, `"Object"` etc.

In [ ]:
from sget.utils.colors import LAYER_STYLES, STYLE_BY_LABEL

# LAYER_STYLES defines the hierarchy order (top to bottom in the UI).
# category_chars is a tuple of all known NodeSymbol prefixes for each layer.
for style in LAYER_STYLES:
    print(
        f"  {style.display_name:12s}  layer_id={style.layer_id:2d}  "
        f"label={style.layer_label:10s}  color={style.color}  "
        f"chars={style.category_chars}"
    )

# Quick lookup by heracles label string.
print(f"\nRoom color: {STYLE_BY_LABEL['Room'].color}")

## Summary

You've learned:
- SGET uses a **model-signal-view** pattern: `SceneGraphModel` is the single source of truth
- The model maintains a **dual store** (Neo4j + in-memory cache) and emits **Qt signals** on every mutation
- `utils/colors.py` is the **single source of truth** for layer configuration — hierarchy order is by position in `LAYER_STYLES`, not raw layer IDs
- Use `heracles.constants` for layer label strings, never hardcode

**Next notebook**: [04_layout_and_navigation.ipynb](04_layout_and_navigation.ipynb) — how nodes are positioned and how to navigate the graph view.